# Deliverable 3 – RPG Battle Simulator

This notebook presents the classes of our mini-RPG battle simulator and shows them in use.

| Module | Class(es) | Purpose |
|---|---|---|
| `rpg_dice.py` | `Dice` | n-sided dice (n ≥ 2) for random damage |
| `rpg_items.py` | `Weapon` | A named weapon that rolls its damage with a `Dice` |
| `rpg_character.py` | `Character`, `Warrior`, `Mage` | Characters with protected HP, plus two specialisations |
| `rpg_battle.py` | `Battle` | Turn-based combat between two characters |

**Game rules**
* Two characters battle until one of them wins.
* Characters can use weapons, where the damage is decided by a dice roll.
* Characters can specialise as a warrior or a mage.
* HP is always between 0 and the character's max HP, and a character dies at 0 HP.

In [1]:
import random
from collections import Counter

from rpg_dice import Dice
from rpg_items import Weapon
from rpg_character import Character, Warrior, Mage
from rpg_battle import Battle

random.seed(201)  # fixed seed so the output is reproducible

## 1. `Dice`

**Attributes:** `sides`
**Methods:** `roll()`, `__repr__()`, `__call__()`

A die needs at least 2 sides and defaults to 6. `__repr__` returns a string that looks like the
code used to create the object, and `__call__` lets the die be rolled by calling it like a function.

In [2]:
d6 = Dice()
d20 = Dice(sides=20)
print(repr(d6), repr(d20))
print("d20.roll():", d20.roll())
print("d20()     :", d20())

Dice(sides=6) Dice(sides=20)
d20.roll(): 3
d20()     : 11


In [3]:
try:
    Dice(1)
except ValueError as error:
    print("ValueError:", error)

ValueError: Dice must have at least 2 sides.


## 2. `Weapon`

**Attributes:** `name`, `damage_dice: Dice`
**Methods:** `roll_damage()`

A weapon *has a* die (composition). The die is passed in, so the same `Weapon` class can model a dagger (d4)
or a great axe (d12).

In [4]:
dagger = Weapon("Dagger", Dice(4))
axe = Weapon("Great Axe", Dice(12))

print(dagger.name, dagger.damage_dice, [dagger.roll_damage() for _ in range(8)])
print(axe.name, axe.damage_dice, [axe.roll_damage() for _ in range(8)])

Dagger Dice(sides=4) [4, 1, 3, 3, 4, 3, 4, 4]
Great Axe Dice(sides=12) [9, 10, 8, 3, 10, 8, 1, 9]


## 3. `Character`

**Attributes:** `name`, `_hp` (protected), `hp_max`, `damage`, `weapon`
**Properties (read-only):** `hp`, `is_alive`
**Methods:** `attack(target)`, `take_damage(damage)`, `heal(amount)`, `introduce()`

`attack()` never touches the target's `_hp` directly. It calls `target.take_damage()`, so each object
is responsible for its own HP. Without a weapon the attack deals `damage`, and with a weapon it rolls the weapon's die.

In [5]:
alice = Character("Alice")
bob = Character("Bob", hp_max=50)
alice.introduce()
print(alice)
print(repr(bob))

Hi, my name is Alice!
Character: Alice with 100 HP and 10 damage
<Character name=Bob hp=50 damage=10>


In [6]:
alice.attack(bob)
print("Bob after one unarmed hit:", bob.hp)

bob.heal(5)
print("Bob after healing 5:", bob.hp)

bob.heal(1000)
print("Bob after healing 1000 (capped at max):", bob.hp, "/", bob.hp_max)

bob.take_damage(1000)
print("Bob after taking 1000 damage (floored at 0):", bob.hp, "- alive?", bob.is_alive)

Bob after one unarmed hit: 40
Bob after healing 5: 45
Bob after healing 1000 (capped at max): 50 / 50
Bob after taking 1000 damage (floored at 0): 0 - alive? False


`is_alive` and `hp` are read-only properties, so they cannot be set from outside the class:

In [7]:
try:
    bob.is_alive = True
except AttributeError as error:
    print("AttributeError:", error)

AttributeError: property 'is_alive' of 'Character' object has no setter


An armed character rolls its weapon's die instead of using its fixed base damage:

In [8]:
knight = Character("Knight", weapon=axe)
dummy = Character("Training Dummy", hp_max=1000)
for _ in range(5):
    before = dummy.hp
    knight.attack(dummy)
    print("Knight hits for", before - dummy.hp)

Knight hits for 4
Knight hits for 4
Knight hits for 12
Knight hits for 4
Knight hits for 1


## 4a. `Warrior(Character)` – defensive specialist

**New attribute:** `armor` (default 5)
**Overrides:** `take_damage()` – reduces incoming damage by `armor` before passing it on to `super().take_damage()`.
Damage can never go below 0, so armor can't heal the warrior.

In [9]:
conan = Warrior("Conan")
tank = Warrior("Tank", armor=50)

dummy.attack(conan)
print("Conan (armor 5) hit for 10 ->", conan.hp)

dummy.attack(tank)
print("Tank (armor 50) hit for 10 ->", tank.hp)

Conan (armor 5) hit for 10 -> 95
Tank (armor 50) hit for 10 -> 100


## 4b. `Mage(Character)` – resource-based attacker

**New attributes:** `mana`, `mana_max` (both default 30)
**Overrides:** `attack()`
* With ≥ 10 mana, the mage spends 10 mana on a spell dealing **double base damage** (any weapon is ignored).
* Otherwise it makes a normal attack (with its weapon, if any) and regains 5 mana (capped at `mana_max`).

In [10]:
merlin = Mage("Merlin", weapon=dagger)
dummy = Character("Training Dummy", hp_max=1000)

for turn in range(1, 8):
    mana_before = merlin.mana
    hp_before = dummy.hp
    merlin.attack(dummy)
    kind = "spell " if mana_before >= 10 else "dagger"
    print(f"Turn {turn}: {kind} dealt {hp_before - dummy.hp:2d} damage, mana {mana_before:2d} -> {merlin.mana:2d}")

Turn 1: spell  dealt 20 damage, mana 30 -> 20
Turn 2: spell  dealt 20 damage, mana 20 -> 10
Turn 3: spell  dealt 20 damage, mana 10 ->  0
Turn 4: dagger dealt  3 damage, mana  0 ->  5
Turn 5: dagger dealt  1 damage, mana  5 -> 10
Turn 6: spell  dealt 20 damage, mana 10 ->  0
Turn 7: dagger dealt  2 damage, mana  0 ->  5


Because spells go through `take_damage()` like every other attack, a warrior's armor also reduces
magic damage. This is polymorphism: `Mage.attack()` doesn't need to know what kind of character it hits.

In [11]:
target = Warrior("Armored Target")
Mage("Apprentice").attack(target)
print("Spell of 20 against armor 5 ->", target.hp)

Spell of 20 against armor 5 -> 85


## 5. `Battle`

**Attributes:** `hero`, `enemy`
**Methods:** `play_round()`, `simulate_battle()`

In each round the hero attacks first. The enemy only strikes back if it is still alive.
`simulate_battle()` repeats rounds until one of them falls.

### Battle 1: Character vs Character, no weapons

In [12]:
random.seed(1)
Battle(Character("Alice"), Character("Bob")).simulate_battle()


--- Round 1 ---
Alice attacks Bob!
Bob has 90 HP left.
Bob attacks Alice!
Alice has 90 HP left.

--- Round 2 ---
Alice attacks Bob!
Bob has 80 HP left.
Bob attacks Alice!
Alice has 80 HP left.

--- Round 3 ---
Alice attacks Bob!
Bob has 70 HP left.
Bob attacks Alice!
Alice has 70 HP left.

--- Round 4 ---
Alice attacks Bob!
Bob has 60 HP left.
Bob attacks Alice!
Alice has 60 HP left.

--- Round 5 ---
Alice attacks Bob!
Bob has 50 HP left.
Bob attacks Alice!
Alice has 50 HP left.

--- Round 6 ---
Alice attacks Bob!
Bob has 40 HP left.
Bob attacks Alice!
Alice has 40 HP left.

--- Round 7 ---
Alice attacks Bob!
Bob has 30 HP left.
Bob attacks Alice!
Alice has 30 HP left.

--- Round 8 ---
Alice attacks Bob!
Bob has 20 HP left.
Bob attacks Alice!
Alice has 20 HP left.

--- Round 9 ---
Alice attacks Bob!
Bob has 10 HP left.
Bob attacks Alice!
Alice has 10 HP left.

--- Round 10 ---
Alice attacks Bob!
Bob has 0 HP left.

Alice wins the battle!


With no weapons and equal stats the damage is fixed, so the hero always wins because it strikes first.

### Battle 2: armed vs unarmed

In [13]:
random.seed(2)
Battle(Character("Peasant"), Character("Knight", weapon=Weapon("Longsword", Dice(20)))).simulate_battle()


--- Round 1 ---
Peasant attacks Knight!
Knight has 90 HP left.
Knight attacks Peasant!
Peasant has 98 HP left.

--- Round 2 ---
Peasant attacks Knight!
Knight has 80 HP left.
Knight attacks Peasant!
Peasant has 95 HP left.

--- Round 3 ---
Peasant attacks Knight!
Knight has 70 HP left.
Knight attacks Peasant!
Peasant has 92 HP left.

--- Round 4 ---
Peasant attacks Knight!
Knight has 60 HP left.
Knight attacks Peasant!
Peasant has 80 HP left.

--- Round 5 ---
Peasant attacks Knight!
Knight has 50 HP left.
Knight attacks Peasant!
Peasant has 74 HP left.

--- Round 6 ---
Peasant attacks Knight!
Knight has 40 HP left.
Knight attacks Peasant!
Peasant has 64 HP left.

--- Round 7 ---
Peasant attacks Knight!
Knight has 30 HP left.
Knight attacks Peasant!
Peasant has 55 HP left.

--- Round 8 ---
Peasant attacks Knight!
Knight has 20 HP left.
Knight attacks Peasant!
Peasant has 35 HP left.

--- Round 9 ---
Peasant attacks Knight!
Knight has 10 HP left.
Knight attacks Peasant!
Peasant has 28 H

### Battle 3: Warrior vs Mage, no weapons

In [14]:
random.seed(3)
Battle(Warrior("Conan"), Mage("Merlin")).simulate_battle()


--- Round 1 ---
Conan attacks Merlin!
Merlin has 90 HP left.
Merlin attacks Conan!
Conan has 85 HP left.

--- Round 2 ---
Conan attacks Merlin!
Merlin has 80 HP left.
Merlin attacks Conan!
Conan has 70 HP left.

--- Round 3 ---
Conan attacks Merlin!
Merlin has 70 HP left.
Merlin attacks Conan!
Conan has 55 HP left.

--- Round 4 ---
Conan attacks Merlin!
Merlin has 60 HP left.
Merlin attacks Conan!
Conan has 50 HP left.

--- Round 5 ---
Conan attacks Merlin!
Merlin has 50 HP left.
Merlin attacks Conan!
Conan has 45 HP left.

--- Round 6 ---
Conan attacks Merlin!
Merlin has 40 HP left.
Merlin attacks Conan!
Conan has 30 HP left.

--- Round 7 ---
Conan attacks Merlin!
Merlin has 30 HP left.
Merlin attacks Conan!
Conan has 25 HP left.

--- Round 8 ---
Conan attacks Merlin!
Merlin has 20 HP left.
Merlin attacks Conan!
Conan has 20 HP left.

--- Round 9 ---
Conan attacks Merlin!
Merlin has 10 HP left.
Merlin attacks Conan!
Conan has 5 HP left.

--- Round 10 ---
Conan attacks Merlin!
Merlin 

### Battle 4: Warrior vs Mage, both armed

In [15]:
random.seed(4)
hero = Warrior("Conan", weapon=Weapon("Great Axe", Dice(12)))
enemy = Mage("Merlin", weapon=Weapon("Staff", Dice(6)))
Battle(hero, enemy).simulate_battle()


--- Round 1 ---
Conan attacks Merlin!
Merlin has 96 HP left.
Merlin attacks Conan!
Conan has 85 HP left.

--- Round 2 ---
Conan attacks Merlin!
Merlin has 91 HP left.
Merlin attacks Conan!
Conan has 70 HP left.

--- Round 3 ---
Conan attacks Merlin!
Merlin has 89 HP left.
Merlin attacks Conan!
Conan has 55 HP left.

--- Round 4 ---
Conan attacks Merlin!
Merlin has 77 HP left.
Merlin attacks Conan!
Conan has 55 HP left.

--- Round 5 ---
Conan attacks Merlin!
Merlin has 69 HP left.
Merlin attacks Conan!
Conan has 55 HP left.

--- Round 6 ---
Conan attacks Merlin!
Merlin has 67 HP left.
Merlin attacks Conan!
Conan has 40 HP left.

--- Round 7 ---
Conan attacks Merlin!
Merlin has 65 HP left.
Merlin attacks Conan!
Conan has 40 HP left.

--- Round 8 ---
Conan attacks Merlin!
Merlin has 58 HP left.
Merlin attacks Conan!
Conan has 40 HP left.

--- Round 9 ---
Conan attacks Merlin!
Merlin has 53 HP left.
Merlin attacks Conan!
Conan has 25 HP left.

--- Round 10 ---
Conan attacks Merlin!
Merlin